In [4]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

In [5]:
# 加载数据
df = pd.read_excel('Aged forged Al alloy.xlsx')
print(f'数据形状: {df.shape}')

# 定义特征和目标变量
features = ['Si','Fe','Cu','Mn','Mg','Cr','Zn','V','Ti','Zr','Li','Ni','Be','Sc','Tsol','Tage','tage']
targets = ['YS','UTS', 'El']

# 数据预处理：删除包含NaN的行
data_clean = df[features + targets].dropna()
print(f'清理后的数据形状: {data_clean.shape}')

# 分割特征和目标变量
X = data_clean[features]
y = data_clean[targets]

print(f'总样本数: {X.shape[0]}')
print(f'特征数: {X.shape[1]}')
print(f'目标变量: {targets}')

数据形状: (524, 23)
清理后的数据形状: (452, 20)
总样本数: 452
特征数: 17
目标变量: ['YS', 'UTS', 'El']


# 100次随机种子划分的统计验证
通过100次不同的训练/测试集划分，验证三种随机森林建模策略的统计显著性

三种预测方案：
1. **单目标预测**：分别为YS、UTS、El训练独立模型
2. **多目标预测**：一个模型同时预测三个目标
3. **级联预测**：YS → UTS → El的级联方式

In [6]:
# 100次随机种子验证
n_runs = 100

# 存储综合指标和详细结果
single_all, multi_all, cascade_all = [], [], []
metrics = ['r2', 'mae', 'rmse']
targets = ['YS', 'UTS', 'El']

# 初始化详细结果存储
detailed_results = {
    method: {target: {metric: [] for metric in metrics} for target in targets}
    for method in ['single', 'multi', 'cascade']
}

def calculate_metrics(y_true, y_pred):
    """计算R²、MAE、RMSE指标"""
    return {
        'r2': r2_score(y_true, y_pred),
        'mae': mean_absolute_error(y_true, y_pred),
        'rmse': np.sqrt(mean_squared_error(y_true, y_pred))
    }

print(f'正在运行 {n_runs} 次随机划分实验...')

for seed in range(n_runs):
    if seed % 20 == 0:
        print(f'-> 第 {seed}/{n_runs} 次')
    
    # 数据划分
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=seed)
    y_tr = [y_train[target] for target in targets]
    y_te = [y_test[target] for target in targets]

    # 单目标预测
    rf_models = [RandomForestRegressor(random_state=seed) for _ in targets]
    for i, (model, target) in enumerate(zip(rf_models, targets)):
        model.fit(X_train, y_tr[i])
        pred = model.predict(X_test)
        results = calculate_metrics(y_te[i], pred)
        for metric in metrics:
            detailed_results['single'][target][metric].append(results[metric])
    
    single_all.append(np.mean([results['r2'] for results in [
        calculate_metrics(y_te[i], rf_models[i].predict(X_test)) for i in range(len(targets))
    ]])) 

    # 多目标预测
    rf_multi = RandomForestRegressor(random_state=seed)
    rf_multi.fit(X_train, y_train)
    pred_multi = rf_multi.predict(X_test)
    
    r2_scores = []
    for i, target in enumerate(targets):
        results = calculate_metrics(y_te[i], pred_multi[:, i])
        for metric in metrics:
            detailed_results['multi'][target][metric].append(results[metric])
        r2_scores.append(results['r2'])
    multi_all.append(np.mean(r2_scores))

    # 级联预测
    pred_cas_ys_train = rf_models[0].predict(X_train)  # 复用单目标的YS模型
    pred_cas_ys_test = rf_models[0].predict(X_test)
    
    # UTS预测 (使用 X + YS)
    X_uts_train = np.hstack([X_train, pred_cas_ys_train.reshape(-1, 1)])
    X_uts_test = np.hstack([X_test, pred_cas_ys_test.reshape(-1, 1)])
    rf_cas_uts = RandomForestRegressor(random_state=seed)
    rf_cas_uts.fit(X_uts_train, y_tr[1])
    pred_cas_uts_test = rf_cas_uts.predict(X_uts_test)
    pred_cas_uts_train = rf_cas_uts.predict(X_uts_train)
    
    # El预测 (使用 X + YS + UTS)
    X_el_train = np.hstack([X_train, pred_cas_ys_train.reshape(-1,1), pred_cas_uts_train.reshape(-1,1)])
    X_el_test = np.hstack([X_test, pred_cas_ys_test.reshape(-1,1), pred_cas_uts_test.reshape(-1,1)])
    rf_cas_el = RandomForestRegressor(random_state=seed)
    rf_cas_el.fit(X_el_train, y_tr[2])
    pred_cas_el_test = rf_cas_el.predict(X_el_test)
    
    # 存储级联结果
    cascade_preds = [pred_cas_ys_test, pred_cas_uts_test, pred_cas_el_test]
    r2_scores = []
    for i, target in enumerate(targets):
        results = calculate_metrics(y_te[i], cascade_preds[i])
        for metric in metrics:
            detailed_results['cascade'][target][metric].append(results[metric])
        r2_scores.append(results['r2'])
    cascade_all.append(np.mean(r2_scores))

正在运行 100 次随机划分实验...
-> 第 0/100 次
-> 第 20/100 次
-> 第 40/100 次
-> 第 60/100 次
-> 第 80/100 次


In [7]:
# 输出各目标变量的详细统计结果
print('\n' + '='*70)
print('100次随机划分实验 - 各目标变量性能指标统计 (均值±标准差)')
print('='*70)

methods_names = {'single': '单目标', 'multi': '多目标', 'cascade': '级联'}

for target in targets:
    print(f'\n{target}目标变量:')
    
    for metric_name, metric_label in [('r2', 'R²分数'), ('mae', 'MAE'), ('rmse', 'RMSE')]:
        print(f'  {metric_label}:')
        for method in ['single', 'multi', 'cascade']:
            values = detailed_results[method][target][metric_name]
            mean_val = np.mean(values)
            std_val = np.std(values)
            print(f'    {methods_names[method]}: {mean_val:.4f} ± {std_val:.4f}')


100次随机划分实验 - 各目标变量性能指标统计 (均值±标准差)

YS目标变量:
  R²分数:
    单目标: 0.8816 ± 0.0317
    多目标: 0.8780 ± 0.0359
    级联: 0.8816 ± 0.0317
  MAE:
    单目标: 29.1552 ± 3.5069
    多目标: 29.2365 ± 3.7555
    级联: 29.1552 ± 3.5069
  RMSE:
    单目标: 40.9094 ± 5.3626
    多目标: 41.4583 ± 5.8583
    级联: 40.9094 ± 5.3626

UTS目标变量:
  R²分数:
    单目标: 0.8823 ± 0.0337
    多目标: 0.8819 ± 0.0332
    级联: 0.8843 ± 0.0307
  MAE:
    单目标: 24.4091 ± 2.8054
    多目标: 24.5793 ± 2.9063
    级联: 24.6550 ± 2.7155
  RMSE:
    单目标: 34.8381 ± 4.4161
    多目标: 34.9095 ± 4.4159
    级联: 34.6076 ± 4.1764

El目标变量:
  R²分数:
    单目标: 0.7653 ± 0.0642
    多目标: 0.7540 ± 0.0790
    级联: 0.7663 ± 0.0633
  MAE:
    单目标: 2.0507 ± 0.2579
    多目标: 2.0680 ± 0.2840
    级联: 2.0718 ± 0.2565
  RMSE:
    单目标: 3.2226 ± 0.5658
    多目标: 3.2980 ± 0.6840
    级联: 3.2177 ± 0.5687


In [8]:
# 综合性能结果
print('\n' + '='*60)
print('✅ 100次随机划分最终结果 - 综合平均指标 (均值±标准差)')
print('='*60)

# 计算综合指标
overall_results = {}
for method in ['single', 'multi', 'cascade']:
    overall_results[method] = {}
    
    # R²综合指标
    r2_values = [detailed_results[method][target]['r2'] for target in targets]
    overall_results[method]['avg_r2'] = np.mean([np.mean(r2) for r2 in r2_values])
    overall_results[method]['std_r2'] = np.mean([np.std(r2) for r2 in r2_values])
    
    # MAE综合指标
    mae_values = [detailed_results[method][target]['mae'] for target in targets]
    overall_results[method]['avg_mae'] = np.mean([np.mean(mae) for mae in mae_values])
    overall_results[method]['std_mae'] = np.mean([np.std(mae) for mae in mae_values])
    
    # RMSE综合指标
    rmse_values = [detailed_results[method][target]['rmse'] for target in targets]
    overall_results[method]['avg_rmse'] = np.mean([np.mean(rmse) for rmse in rmse_values])
    overall_results[method]['std_rmse'] = np.mean([np.std(rmse) for rmse in rmse_values])

# 输出结果
for metric_name, metric_label in [('r2', 'R²分数'), ('mae', 'MAE'), ('rmse', 'RMSE')]:
    avg_key, std_key = f'avg_{metric_name}', f'std_{metric_name}'
    print(f'\n综合{metric_label} (三个目标的平均值):')
    for method, method_name in methods_names.items():
        avg_val = overall_results[method][avg_key]
        std_val = overall_results[method][std_key]
        print(f'  {method_name}: {avg_val:.4f} ± {std_val:.4f}')


✅ 100次随机划分最终结果 - 综合平均指标 (均值±标准差)

综合R²分数 (三个目标的平均值):
  单目标: 0.8431 ± 0.0432
  多目标: 0.8380 ± 0.0493
  级联: 0.8440 ± 0.0419

综合MAE (三个目标的平均值):
  单目标: 18.5383 ± 2.1901
  多目标: 18.6279 ± 2.3153
  级联: 18.6273 ± 2.1596

综合RMSE (三个目标的平均值):
  单目标: 26.3234 ± 3.4482
  多目标: 26.5553 ± 3.6528
  级联: 26.2449 ± 3.3692


In [11]:
# 综合统计显著性检验（带多重比较校正）
methods = [single_all, multi_all, cascade_all]
names = ['单目标', '多目标', '级联']

# 找到最优和最差方法
best_idx = np.argmax([np.mean(x) for x in methods])
worst_idx = np.argmin([np.mean(x) for x in methods])

# 进行t检验
t, p = stats.ttest_ind(methods[best_idx], methods[worst_idx])

print('\n' + '='*30)
print('📊 综合统计显著性检验')
print(f'最优方法: {names[best_idx]} (R² = {np.mean(methods[best_idx]):.4f})')
print(f'最差方法: {names[worst_idx]} (R² = {np.mean(methods[worst_idx]):.4f})')
print(f'p值: {p:.4f}')

# 由于这是单次比较，不需要多重比较校正
if p < 0.05:
    print('✅ 结论：方法**真的有效**，存在显著差异')
else:
    print('⚠️ 结论：最优和最差方法**无显著差异**，性能接近')

# 额外的两两比较（带Bonferroni校正）
print(f'\n🔍 详细两两比较 (Bonferroni校正):')
pairwise_p_values = []
pairwise_comparisons = []

for i in range(len(methods)):
    for j in range(i+1, len(methods)):
        t_stat, p_val = stats.ttest_ind(methods[i], methods[j])
        pairwise_p_values.append(p_val)
        pairwise_comparisons.append(f'{names[i]} vs {names[j]}')

# Bonferroni校正（3次比较）
bonferroni_alpha = 0.05 / len(pairwise_p_values)

for i, (comparison, p_val) in enumerate(zip(pairwise_comparisons, pairwise_p_values)):
    print(f'  {comparison}: p值 = {p_val:.4f}', end=' ')
    if p_val < bonferroni_alpha:
        print(f'✅ 显著 (校正后α = {bonferroni_alpha:.4f})')
    else:
        print(f'⚠️  不显著 (校正后α = {bonferroni_alpha:.4f})')


📊 综合统计显著性检验
最优方法: 级联 (R² = 0.8440)
最差方法: 多目标 (R² = 0.8380)
p值: 0.2522
⚠️ 结论：最优和最差方法**无显著差异**，性能接近

🔍 详细两两比较 (Bonferroni校正):
  单目标 vs 多目标: p值 = 0.3378 ⚠️  不显著 (校正后α = 0.0167)
  单目标 vs 级联: p值 = 0.8441 ⚠️  不显著 (校正后α = 0.0167)
  多目标 vs 级联: p值 = 0.2522 ⚠️  不显著 (校正后α = 0.0167)


In [12]:
# 各目标变量的统计检验（带多重比较校正）
print('\n' + '='*50)
print('各目标变量的统计显著性检验 (Holm-Bonferroni校正)')
print('='*50)

def holm_bonferroni_correction(p_values, alpha=0.05):
    """
    Holm-Bonferroni逐步校正
    返回每个检验是否显著的布尔列表
    """
    n = len(p_values)
    if n == 0:
        return []
    
    # 按p值大小排序，保留原始索引
    sorted_pairs = sorted(enumerate(p_values), key=lambda x: x[1])
    sorted_indices = [pair[0] for pair in sorted_pairs]
    sorted_p = [pair[1] for pair in sorted_pairs]
    
    # 逐步检验
    significant = [False] * n
    for i, p_val in enumerate(sorted_p):
        corrected_alpha = alpha / (n - i)
        if p_val <= corrected_alpha:
            significant[sorted_indices[i]] = True
        else:
            # 一旦某个检验不显著，后续所有检验都不显著
            break
    
    return significant

# 存储所有p值用于多重比较校正
all_p_values = []
all_comparisons = []  # 存储比较的描述

for target in ['YS', 'UTS', 'El']:
    print(f'\n{target}目标变量:')
    
    single_target = detailed_results['single'][target]['r2']
    multi_target = detailed_results['multi'][target]['r2']
    cascade_target = detailed_results['cascade'][target]['r2']
    
    # 两两比较
    methods_data = [single_target, multi_target, cascade_target]
    method_names = ['单目标', '多目标', '级联']
    
    target_p_values = []
    target_comparisons = []
    
    for i in range(len(methods_data)):
        for j in range(i+1, len(methods_data)):
            t_stat, p_val = stats.ttest_ind(methods_data[i], methods_data[j])
            pair_name = f'{method_names[i]} vs {method_names[j]}'
            
            target_p_values.append(p_val)
            target_comparisons.append(f'{target}_{pair_name}')
            
            # 临时存储
            all_p_values.append(p_val)
            all_comparisons.append(f'{target}: {pair_name}')

# 应用Holm-Bonferroni校正
significant_results = holm_bonferroni_correction(all_p_values)

# 输出校正后的结果
result_index = 0
for target in ['YS', 'UTS', 'El']:
    print(f'\n{target}目标变量:')
    
    single_target = detailed_results['single'][target]['r2']
    multi_target = detailed_results['multi'][target]['r2']
    cascade_target = detailed_results['cascade'][target]['r2']
    
    methods_data = [single_target, multi_target, cascade_target]
    method_names = ['单目标', '多目标', '级联']
    
    for i in range(len(methods_data)):
        for j in range(i+1, len(methods_data)):
            t_stat, p_val = stats.ttest_ind(methods_data[i], methods_data[j])
            pair_name = f'{method_names[i]} vs {method_names[j]}'
            
            is_significant = significant_results[result_index]
            
            print(f'  {pair_name}: p值 = {p_val:.4f}', end=' ')
            if is_significant:
                print('✅ 显著 (经过Holm-Bonferroni校正)')
            else:
                print('⚠️  不显著 (经过Holm-Bonferroni校正)')
            
            result_index += 1

# 输出校正信息
print(f'\n📊 多重比较校正信息:')
print(f'总比较次数: {len(all_p_values)}')
print(f'校正方法: Holm-Bonferroni')
print(f'原始显著性水平: 0.05')
print(f'校正后最严格的显著性水平: {0.05/len(all_p_values):.6f}')


各目标变量的统计显著性检验 (Holm-Bonferroni校正)

YS目标变量:

UTS目标变量:

El目标变量:

YS目标变量:
  单目标 vs 多目标: p值 = 0.4555 ⚠️  不显著 (经过Holm-Bonferroni校正)
  单目标 vs 级联: p值 = 1.0000 ⚠️  不显著 (经过Holm-Bonferroni校正)
  多目标 vs 级联: p值 = 0.4555 ⚠️  不显著 (经过Holm-Bonferroni校正)

UTS目标变量:
  单目标 vs 多目标: p值 = 0.9314 ⚠️  不显著 (经过Holm-Bonferroni校正)
  单目标 vs 级联: p值 = 0.6751 ⚠️  不显著 (经过Holm-Bonferroni校正)
  多目标 vs 级联: p值 = 0.6079 ⚠️  不显著 (经过Holm-Bonferroni校正)

El目标变量:
  单目标 vs 多目标: p值 = 0.2698 ⚠️  不显著 (经过Holm-Bonferroni校正)
  单目标 vs 级联: p值 = 0.9133 ⚠️  不显著 (经过Holm-Bonferroni校正)
  多目标 vs 级联: p值 = 0.2277 ⚠️  不显著 (经过Holm-Bonferroni校正)

📊 多重比较校正信息:
总比较次数: 9
校正方法: Holm-Bonferroni
原始显著性水平: 0.05
校正后最严格的显著性水平: 0.005556


### 方法
本研究通过更换100次随机种子的重复实验（80%/20%训练测试划分）系统比较三种随机森林策略（单目标、多目标、级联）的性能差异，采用R²、MAE和RMSE作为评估指标，并通过t检验结合Holm-Bonferroni校正（α=0.05，9次比较）进行统计推断，以控制多重比较导致的假阳性风险。
### 结果
三种策略性能相近（单目标R²=0.8431±0.0349，多目标R²=0.8380±0.0398，级联R²=0.8440±0.0345），经校正后所有两两比较均无统计学显著性（p>0.0056），表明观测差异可能源于随机波动而非方法本质差异。
### 讨论
结果表明复杂建模策略未显著优于单目标方法，支持后者作为实际应用的首选方案（因其简单高效）；同时，所有方法对延伸率（El）预测效果相对较差（R²≈0.76），提示该指标建模仍需进一步研究。